In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv) 

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Import all the necessary libraries:

In [6]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error
from scipy.sparse import hstack
import lightgbm as lgb

In [7]:

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


Load train and test data:

In [8]:


train = pd.read_csv( 'train.csv')
test = pd.read_csv( 'test.csv')

In [9]:
for col in train.columns:
    print(col)

sample_id
catalog_content
image_link
price


Utility Functions:

In [10]:
# unit_map = {
#     "ml": "ml", "milliliter": "ml", "milliliters": "ml", "millilitres": "ml", "millilitres": "ml",
#     "l": "l", "liter": "l", "litre": "l",
#     "oz": "oz", "ounce": "oz", "ounces": "oz",
#     "fl oz": "fl_oz", "fluid ounce": "fl_oz", "fluid ounces": "fl_oz",
#     "g": "g", "gram": "g", "grams": "g", "gm": "g",
#     "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo gram": "kg", "kilo grams": "kg",
#     "count": "count", "ct": "count", "pcs": "count", "piece": "count", "pack": "pack"
# }

# keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant"]

# def clean_text(text):
#     """Basic text cleaning for TF-IDF"""
#     text = re.sub(r"â€“|â€|Ã|™", "", str(text))
#     text = str(text).lower()
#     text = re.sub(r'[^a-z0-9\s\.\%\-]+', ' ', text)
#     text = re.sub(r'\s+', ' ', text).strip()
#     return text

# def extract_value(text):
#     match = re.search(r"Value:\s*([\d\.]+)", str(text))
#     return float(match.group(1)) if match else np.nan

# def extract_unit(text):
#     text = str(text).lower()
#     for pattern, unit in unit_map.items():
#         if re.search(rf'\b{pattern}\b', text):
#             return unit
#     return "unknown"

# def strip_item_name(text):
#     return re.sub(r'\bitem name\b', '', str(text), flags=re.IGNORECASE).strip()

# def feature_engineering(df):
#     """Feature engineering from catalog content"""
#     df["catalog_content"] = df["catalog_content"].fillna("").apply(clean_text)
#     df["Value"] = df["catalog_content"].apply(extract_value)
#     df["Unit"] = df["catalog_content"].apply(extract_unit)
#     df["Item_Name"] = df["catalog_content"].apply(strip_item_name)

#     # Fill missing numeric features
#     df["Value"] = df["Value"].fillna(df["Value"].median())
#     df["Unit"] = df["Unit"].fillna("unknown")



#     df["Value"] = pd.to_numeric(df["Value"], errors="coerce").fillna(0)
#     df["Value"] = df["Value"].clip(lower=0)
#     df["log_value"] = np.log1p(df["Value"])
#     df["word_count"] = df["catalog_content"].apply(lambda x: len(x.split()))
#     df["char_count"] = df["catalog_content"].apply(len)
#     df["digit_ratio"] = df["catalog_content"].apply(lambda x: sum(c.isdigit() for c in x) / max(len(x), 1))
#     df["avg_word_len"] = df["catalog_content"].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) else 0)

#     df["bullet_count"] = df["catalog_content"].str.count("bullet point")


#     # Keyword flags (you can tune/add more keywords as you analyze data)
#     for kw in keywords:
#         df[f"has_{kw}"] = df["catalog_content"].str.contains(kw, case=False).astype(int)

#     return df

In [11]:
import re
import numpy as np
import pandas as pd

# Clean text helper
def clean_text(text):
    text = re.sub(r"â€“|â€|Ã|™", "", str(text))
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s\.\%\-]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_value(text):
    match = re.search(r"Value:\s*([\d\.]+)", str(text))
    return float(match.group(1)) if match else np.nan

def extract_unit(text):
    unit_map = {
        "ml": "ml", "milliliter": "ml", "milliliters": "ml", "millilitres": "ml",
        "l": "l", "liter": "l", "litre": "l",
        "oz": "oz", "ounce": "oz", "ounces": "oz",
        "fl oz": "fl_oz", "fluid ounce": "fl_oz", "fluid ounces": "fl_oz",
        "g": "g", "gram": "g", "grams": "g", "gm": "g",
        "kg": "kg", "kilogram": "kg", "kilograms": "kg", "kilo gram": "kg", "kilo grams": "kg",
        "count": "count", "ct": "count", "pcs": "count", "piece": "count", "pack": "pack"
    }
    text = str(text).lower()
    for pattern, unit in unit_map.items():
        if re.search(rf'\b{pattern}\b', text):
            return unit
    return "unknown"

def extract_item_name(text):
    match = re.search(r"Item Name:\s*(.*?)(?=Bullet Point:|Product Description:|Value:|Unit:|$)", str(text), flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""

def extract_bullet_points(text):
    # Find all bullet points and join them into one string
    bullets = re.findall(r"Bullet Point\s*\d*:\s*(.*?)(?=Bullet Point|Item Name:|Product Description:|Value:|Unit:|$)", str(text), flags=re.DOTALL | re.IGNORECASE)
    return " ".join([b.strip() for b in bullets])

def extract_product_description(text):
    match = re.search(r"Product Description:\s*(.*?)(?=Item Name:|Bullet Point:|Value:|Unit:|$)", str(text), flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""

keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant",
    "new", "fresh", "natural", "handmade", "limited", "sale", "discount", "special", "gift", "free",
    "imported", "authentic", "original", "pure", "light", "soft", "strong", "durable", "high-quality",
    "bestseller", "top-rated", "popular", "classic", "eco-friendly", "recyclable", "non-toxic", "safe",
    "baby", "kids", "adult", "men", "women", "unisex", "winter", "summer", "travel", "portable",
    "premium-grade", "luxury", "compact", "multi-purpose", "versatile", "handcrafted", "exclusive"]

def keyword_flag(text, kw):
    # Negative patterns
    neg_patterns = [r'\bno\s+', r'\bnot\s+', r'\bnon\s+', r'\bwithout\s+']
    # If any negation appears before keyword, return 0
    for neg in neg_patterns:
        if re.search(neg + kw, text, flags=re.IGNORECASE):
            return 0
    # Otherwise, if keyword exists
    return 1 if re.search(rf'\b{kw}\b', text, flags=re.IGNORECASE) else 0
def feature_engineering(df, keywords=None):
    
    keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant",
    "new", "fresh", "natural", "handmade", "limited", "sale", "discount", "special", "gift", "free",
    "imported", "authentic", "original", "pure", "light", "soft", "strong", "durable", "high-quality",
    "bestseller", "top-rated", "popular", "classic", "eco-friendly", "recyclable", "non-toxic", "safe",
    "baby", "kids", "adult", "men", "women", "unisex", "winter", "summer", "travel", "portable",
    "premium-grade", "luxury", "compact", "multi-purpose", "versatile", "handcrafted", "exclusive"]

    
    """Feature engineering from catalog content with separate columns."""
    df["catalog_content"] = df["catalog_content"].fillna("")

    # Extract structured sections
    df["item_name_raw"] = df["catalog_content"].apply(extract_item_name)
    df["bullet_points_raw"] = df["catalog_content"].apply(extract_bullet_points)
    df["product_desc_raw"] = df["catalog_content"].apply(extract_product_description)
    df["value_raw"] = df["catalog_content"].apply(extract_value)
    df["unit_raw"] = df["catalog_content"].apply(extract_unit)

    # Clean text columns
    df["item_name"] = df["item_name_raw"].apply(clean_text)
    df["bullet_points"] = df["bullet_points_raw"].apply(clean_text)
    df["product_desc"] = df["product_desc_raw"].apply(clean_text)

    # Fill missing numeric
    df["value_raw"] = df["value_raw"].fillna(df["value_raw"].median())
    df["unit_raw"] = df["unit_raw"].fillna("unknown")

    # Numeric / derived features
    df["log_value"] = np.log1p(df["value_raw"])
    df["bullet_point_count"] = df["bullet_points_raw"].apply(lambda x: len(re.findall(r'\.', x)))  # approximate count
    df["word_count_item"] = df["item_name"].apply(lambda x: len(x.split()))
    df["word_count_bullet"] = df["bullet_points"].apply(lambda x: len(x.split()))
    df["word_count_desc"] = df["product_desc"].apply(lambda x: len(x.split()))
    df["word_count_total"] = df["word_count_item"] + df["word_count_bullet"] + df["word_count_desc"]
    df["char_count_total"] = df["item_name"].apply(len) + df["bullet_points"].apply(len) + df["product_desc"].apply(len)
    df["digit_ratio_total"] = df["item_name"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)) + \
                              df["bullet_points"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)) + \
                              df["product_desc"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1))
    df["avg_word_len_total"] = df["char_count_total"] / df["word_count_total"].replace(0,1)

    # Keyword flags
    
    for kw in keywords:
        df[f"has_{kw}"] = df["catalog_content"].apply(lambda x: keyword_flag(x, kw))

    return df

# Enhanced Feature Engineering Strategy

We'll implement several improvements to capture better patterns:

1. **Unit Categorization**: Group units into logical categories (weight, volume, count, etc.)
2. **Price-Unit Relationships**: Extract price per unit patterns
3. **Product Categories**: Infer product categories from text content
4. **Feature Correlations**: Analyze and create interaction features
5. **Category-Specific Features**: Features tailored to different product types
6. **Brand/Manufacturer Extraction**: Extract brand information
7. **Quality/Premium Indicators**: Detect premium product signals

In [ ]:
# Enhanced Feature Engineering with Better Categorization and Correlation Analysis

import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Enhanced unit mapping with categories
UNIT_CATEGORIES = {
    'weight': ['g', 'gram', 'grams', 'gm', 'kg', 'kilogram', 'kilograms', 'kilo gram', 'kilo grams', 
               'lb', 'lbs', 'pound', 'pounds', 'ton', 'tonnes'],
    'volume': ['ml', 'milliliter', 'milliliters', 'millilitres', 'l', 'liter', 'litre', 'litres', 'liters',
               'fl oz', 'fluid ounce', 'fluid ounces', 'gallon', 'gallons', 'quart', 'quarts', 'pint', 'pints'],
    'count': ['count', 'ct', 'pcs', 'piece', 'pieces', 'pc', 'units', 'unit', 'items', 'item'],
    'pack': ['pack', 'packs', 'package', 'packages', 'box', 'boxes', 'set', 'sets', 'bundle', 'bundles'],
    'length': ['cm', 'centimeter', 'centimeters', 'm', 'meter', 'meters', 'mm', 'millimeter', 'millimeters',
               'inch', 'inches', 'ft', 'feet', 'foot', 'yard', 'yards'],
    'area': ['sq ft', 'square feet', 'sq m', 'square meter', 'sq cm', 'square cm'],
    'other': ['unknown', 'na', 'none']
}

# Create reverse mapping
UNIT_TO_CATEGORY = {}
for category, units in UNIT_CATEGORIES.items():
    for unit in units:
        UNIT_TO_CATEGORY[unit] = category

# Product category keywords
PRODUCT_CATEGORIES = {
    'food_beverage': ['food', 'drink', 'beverage', 'snack', 'coffee', 'tea', 'juice', 'water', 'milk', 'chocolate', 
                     'candy', 'cereal', 'bread', 'pasta', 'rice', 'flour', 'sugar', 'salt', 'spice', 'sauce',
                     'oil', 'vinegar', 'honey', 'jam', 'butter', 'cheese', 'yogurt', 'ice cream'],
    'health_beauty': ['cream', 'lotion', 'shampoo', 'soap', 'toothpaste', 'vitamin', 'supplement', 'medicine',
                     'skincare', 'cosmetic', 'makeup', 'perfume', 'deodorant', 'sunscreen', 'moisturizer'],
    'home_kitchen': ['kitchen', 'cookware', 'utensil', 'plate', 'cup', 'bowl', 'pan', 'pot', 'knife', 'spoon',
                    'fork', 'towel', 'napkin', 'cleaner', 'detergent', 'tissue', 'paper', 'bag', 'container'],
    'clothing_accessories': ['shirt', 'pants', 'dress', 'shoe', 'sock', 'hat', 'belt', 'watch', 'jewelry', 'bag',
                            'wallet', 'sunglasses', 'scarf', 'gloves', 'jacket', 'coat', 'sweater'],
    'electronics': ['electronic', 'battery', 'charger', 'cable', 'phone', 'computer', 'laptop', 'tablet', 'camera',
                   'speaker', 'headphone', 'tv', 'remote', 'gaming', 'console'],
    'baby_kids': ['baby', 'infant', 'toddler', 'kids', 'children', 'diaper', 'formula', 'toy', 'game', 'book',
                 'cradle', 'stroller', 'car seat'],
    'sports_outdoors': ['sports', 'fitness', 'outdoor', 'camping', 'hiking', 'fishing', 'hunting', 'bicycle',
                       'exercise', 'gym', 'yoga', 'running', 'swimming'],
    'automotive': ['car', 'auto', 'vehicle', 'tire', 'oil', 'filter', 'brake', 'engine', 'battery', 'automotive'],
    'office_supplies': ['office', 'pen', 'pencil', 'paper', 'notebook', 'folder', 'stapler', 'printer', 'ink']
}

# Brand extraction patterns
BRAND_PATTERNS = [
    r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b(?=\s+(?:brand|by|from))',  # Brand indicators
    r'(?:by|from)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)',  # "by Brand" or "from Brand"
    r'\b([A-Z]{2,})\b',  # All caps (likely brand acronyms)
]

def extract_brand(text):
    """Extract potential brand names from text"""
    text = str(text)
    brands = []
    
    for pattern in BRAND_PATTERNS:
        matches = re.findall(pattern, text)
        brands.extend(matches)
    
    # Return most frequent brand or first found
    if brands:
        brand_counts = Counter(brands)
        return brand_counts.most_common(1)[0][0]
    return "unknown"

def get_unit_category(unit):
    """Map unit to category"""
    unit = str(unit).lower()
    return UNIT_TO_CATEGORY.get(unit, 'other')

def get_product_category(text):
    """Infer product category from text content"""
    text = str(text).lower()
    category_scores = {}
    
    for category, keywords in PRODUCT_CATEGORIES.items():
        score = sum(1 for keyword in keywords if keyword in text)
        category_scores[category] = score
    
    # Return category with highest score, or 'other' if no matches
    if category_scores and max(category_scores.values()) > 0:
        return max(category_scores.items(), key=lambda x: x[1])[0]
    return 'other'

def extract_numeric_values(text):
    """Extract all numeric values from text"""
    numbers = re.findall(r'\d+\.?\d*', str(text))
    return [float(n) for n in numbers]

def get_premium_indicators(text):
    """Count premium/quality indicators"""
    premium_words = ['premium', 'luxury', 'high-quality', 'professional', 'deluxe', 'supreme', 
                    'superior', 'exclusive', 'limited', 'special', 'artisan', 'gourmet', 'organic']
    text = str(text).lower()
    return sum(1 for word in premium_words if word in text)

def get_discount_indicators(text):
    """Count discount/sale indicators"""
    discount_words = ['sale', 'discount', 'cheap', 'budget', 'economy', 'value', 'affordable', 'clearance']
    text = str(text).lower()
    return sum(1 for word in discount_words if word in text)

In [ ]:
def enhanced_feature_engineering(df, is_train=True, correlation_analysis=False):
    """
    Enhanced feature engineering with categorization and correlation analysis
    """
    print("Starting enhanced feature engineering...")
    
    # Basic text cleaning and extraction (keeping existing functions)
    df["catalog_content"] = df["catalog_content"].fillna("")
    df["item_name_raw"] = df["catalog_content"].apply(extract_item_name)
    df["bullet_points_raw"] = df["catalog_content"].apply(extract_bullet_points)
    df["product_desc_raw"] = df["catalog_content"].apply(extract_product_description)
    df["value_raw"] = df["catalog_content"].apply(extract_value)
    df["unit_raw"] = df["catalog_content"].apply(extract_unit)
    
    # Clean text columns
    df["item_name"] = df["item_name_raw"].apply(clean_text)
    df["bullet_points"] = df["bullet_points_raw"].apply(clean_text)
    df["product_desc"] = df["product_desc_raw"].apply(clean_text)
    df["combined_text"] = df["item_name"] + " " + df["bullet_points"] + " " + df["product_desc"]
    
    # === NEW ENHANCED FEATURES ===
    
    # 1. Unit Categorization
    df["unit_category"] = df["unit_raw"].apply(get_unit_category)
    
    # 2. Product Category Detection
    df["product_category"] = df["combined_text"].apply(get_product_category)
    
    # 3. Brand Extraction
    df["brand"] = df["combined_text"].apply(extract_brand)
    
    # 4. Premium and Discount Indicators
    df["premium_score"] = df["combined_text"].apply(get_premium_indicators)
    df["discount_score"] = df["combined_text"].apply(get_discount_indicators)
    
    # 5. Numeric Value Extraction
    df["numeric_values"] = df["combined_text"].apply(extract_numeric_values)
    df["numeric_count"] = df["numeric_values"].apply(len)
    df["max_numeric"] = df["numeric_values"].apply(lambda x: max(x) if x else 0)
    df["min_numeric"] = df["numeric_values"].apply(lambda x: min(x) if x else 0)
    df["avg_numeric"] = df["numeric_values"].apply(lambda x: np.mean(x) if x else 0)
    
    # 6. Enhanced Text Features
    df["exclamation_count"] = df["combined_text"].str.count('!')
    df["question_count"] = df["combined_text"].str.count(r'\?')
    df["uppercase_ratio"] = df["combined_text"].apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    df["punctuation_density"] = df["combined_text"].apply(lambda x: sum(1 for c in x if c in '.,!?;:') / max(len(x), 1))
    
    # 7. Size/Quantity Features
    df["has_multi_pack"] = df["combined_text"].str.contains(r'\d+\s*pack|\d+\s*ct|\d+\s*pcs', case=False, regex=True).astype(int)
    df["pack_size"] = df["combined_text"].str.extract(r'(\d+)\s*(?:pack|ct|pcs)', expand=False).fillna(0).astype(float)
    
    # Fill missing values
    df["value_raw"] = df["value_raw"].fillna(df["value_raw"].median() if not df["value_raw"].isna().all() else 0)
    df["unit_raw"] = df["unit_raw"].fillna("unknown")
    
    # 8. Basic numerical features (keeping existing ones)
    df["log_value"] = np.log1p(df["value_raw"])
    df["bullet_point_count"] = df["bullet_points_raw"].apply(lambda x: len(re.findall(r'\.', x)))
    df["word_count_item"] = df["item_name"].apply(lambda x: len(x.split()))
    df["word_count_bullet"] = df["bullet_points"].apply(lambda x: len(x.split()))
    df["word_count_desc"] = df["product_desc"].apply(lambda x: len(x.split()))
    df["word_count_total"] = df["word_count_item"] + df["word_count_bullet"] + df["word_count_desc"]
    df["char_count_total"] = df["item_name"].apply(len) + df["bullet_points"].apply(len) + df["product_desc"].apply(len)
    df["digit_ratio_total"] = (df["item_name"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)) + 
                              df["bullet_points"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)) + 
                              df["product_desc"].apply(lambda x: sum(c.isdigit() for c in x)/max(len(x),1)))
    df["avg_word_len_total"] = df["char_count_total"] / df["word_count_total"].replace(0,1)
    
    # 9. Category-specific features
    df["is_food_beverage"] = (df["product_category"] == 'food_beverage').astype(int)
    df["is_health_beauty"] = (df["product_category"] == 'health_beauty').astype(int)
    df["is_electronics"] = (df["product_category"] == 'electronics').astype(int)
    
    # 10. Unit-category specific features
    df["is_weight_unit"] = (df["unit_category"] == 'weight').astype(int)
    df["is_volume_unit"] = (df["unit_category"] == 'volume').astype(int)
    df["is_count_unit"] = (df["unit_category"] == 'count').astype(int)
    
    # 11. Interaction features
    df["value_x_premium"] = df["value_raw"] * df["premium_score"]
    df["word_count_x_premium"] = df["word_count_total"] * df["premium_score"]
    df["pack_size_x_value"] = df["pack_size"] * df["value_raw"]
    
    # 12. Keyword flags (enhanced with negation handling)
    enhanced_keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant",
                        "new", "fresh", "natural", "handmade", "limited", "sale", "discount", "special", "gift", "free",
                        "imported", "authentic", "original", "pure", "light", "soft", "strong", "durable", "high-quality",
                        "bestseller", "top-rated", "popular", "classic", "eco-friendly", "recyclable", "non-toxic", "safe",
                        "baby", "kids", "adult", "men", "women", "unisex", "winter", "summer", "travel", "portable",
                        "premium-grade", "luxury", "compact", "multi-purpose", "versatile", "handcrafted", "exclusive"]
    
    for kw in enhanced_keywords:
        df[f"has_{kw}"] = df["combined_text"].apply(lambda x: keyword_flag(x, kw))
    
    print(f"Enhanced feature engineering complete. Added {len([col for col in df.columns if col not in ['sample_id', 'catalog_content', 'price']])} features.")
    
    # Correlation analysis for training data
    if is_train and correlation_analysis and 'price' in df.columns:
        print("Performing correlation analysis...")
        analyze_feature_correlations(df)
    
    return df

def analyze_feature_correlations(df, top_n=20):
    """Analyze correlations between features and target price"""
    
    # Get numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'sample_id' in numeric_cols:
        numeric_cols.remove('sample_id')
    
    if 'price' not in df.columns:
        print("Price column not found. Skipping correlation analysis.")
        return
    
    # Calculate correlations with price
    correlations = df[numeric_cols].corr()['price'].abs().sort_values(ascending=False)
    
    print(f"\nTop {top_n} Features Correlated with Price:")
    print("=" * 50)
    for feature, corr in correlations.head(top_n).items():
        if feature != 'price':
            print(f"{feature:<30}: {corr:.4f}")
    
    # Category analysis
    print(f"\nPrice Distribution by Product Category:")
    print("=" * 50)
    cat_stats = df.groupby('product_category')['price'].agg(['count', 'mean', 'median', 'std']).round(2)
    print(cat_stats)
    
    print(f"\nPrice Distribution by Unit Category:")
    print("=" * 50)
    unit_stats = df.groupby('unit_category')['price'].agg(['count', 'mean', 'median', 'std']).round(2)
    print(unit_stats)
    
    return correlations

Feature Engineering:

In [ ]:
# Apply Enhanced Feature Engineering
print("Applying enhanced feature engineering to training data...")
train = enhanced_feature_engineering(train, is_train=True, correlation_analysis=True)

print("\nApplying enhanced feature engineering to test data...")
test = enhanced_feature_engineering(test, is_train=False, correlation_analysis=False)

# Encode categorical features
categorical_features = ['unit_raw', 'unit_category', 'product_category', 'brand']

label_encoders = {}
for feature in categorical_features:
    le = LabelEncoder()
    
    # Combine train and test for consistent encoding
    combined_values = pd.concat([train[feature], test[feature]]).astype(str).fillna('unknown')
    le.fit(combined_values)
    
    train[f"{feature}_enc"] = le.transform(train[feature].astype(str).fillna('unknown'))
    test[f"{feature}_enc"] = le.transform(test[feature].astype(str).fillna('unknown'))
    
    label_encoders[feature] = le
    print(f"Encoded {feature}: {len(le.classes_)} unique values")

print("\nFeature engineering complete!")

TF-IDF Features:

In [13]:
# tfidf = TfidfVectorizer(
#     max_features=30000,
#     ngram_range=(1, 2),
#     min_df=3,
#     max_df=0.9,
#     sublinear_tf=True,
#     stop_words=None,
#     analyzer='word'
# )
# # tfidf_name = TfidfVectorizer(
# #     max_features=5000,
# #     ngram_range=(1, 2),
# #     min_df=1,
# #     max_df=0.95,
# #     sublinear_tf=True,
# #     stop_words=None,
# #     analyzer='word'
# # )

# # Clean text just to be safe
# # train["Item_Name"] = train["Item_Name"].fillna("").astype(str)
# # test["Item_Name"] = test["Item_Name"].fillna("").astype(str)

# train["catalog_content"] = train["catalog_content"].fillna("").astype(str)
# test["catalog_content"] = test["catalog_content"].fillna("").astype(str)

# tfidf_train = tfidf.fit_transform(train["catalog_content"])
# tfidf_test = tfidf.transform(test["catalog_content"])
# # Item_train = tfidf_name.fit_transform(train["Item_Name"])
# # Item_test = tfidf_name.transform(test["Item_Name"])

In [ ]:
for col in ["item_name", "bullet_points", "product_desc", "combined_text"]:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)

from sklearn.feature_extraction.text import TfidfVectorizer

# Base TF-IDF parameters
tfidf_params = dict(
    max_features=20000,  # Reduced to manage memory
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    analyzer='word'
)

print("Creating TF-IDF features...")

# 1. Individual text columns
tfidf_item = TfidfVectorizer(**tfidf_params)
tfidf_item_train = tfidf_item.fit_transform(train["item_name"])
tfidf_item_test = tfidf_item.transform(test["item_name"])

tfidf_bullet = TfidfVectorizer(**tfidf_params)
tfidf_bullet_train = tfidf_bullet.fit_transform(train["bullet_points"])
tfidf_bullet_test = tfidf_bullet.transform(test["bullet_points"])

tfidf_desc = TfidfVectorizer(**tfidf_params)
tfidf_desc_train = tfidf_desc.fit_transform(train["product_desc"])
tfidf_desc_test = tfidf_desc.transform(test["product_desc"])

# 2. Category-specific TF-IDF for major categories
major_categories = ['food_beverage', 'health_beauty', 'electronics', 'home_kitchen']
category_tfidf_features = []

for category in major_categories:
    print(f"Creating TF-IDF for category: {category}")
    
    # Filter data for this category
    train_cat = train[train['product_category'] == category]['combined_text']
    test_cat = test[test['product_category'] == category]['combined_text']
    
    if len(train_cat) > 10:  # Only if we have enough samples
        # Category-specific TF-IDF with smaller feature set
        cat_tfidf = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1,2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        
        # Fit on category data
        cat_tfidf.fit(train_cat)
        
        # Transform all data (will be zero for non-category items)
        cat_train_features = cat_tfidf.transform(train["combined_text"])
        cat_test_features = cat_tfidf.transform(test["combined_text"])
        
        # Zero out features for items not in this category
        cat_mask_train = (train['product_category'] == category).values
        cat_mask_test = (test['product_category'] == category).values
        
        cat_train_features = cat_train_features.multiply(cat_mask_train.reshape(-1, 1))
        cat_test_features = cat_test_features.multiply(cat_mask_test.reshape(-1, 1))
        
        category_tfidf_features.append((cat_train_features, cat_test_features))

print("Stacking TF-IDF features...")
from scipy.sparse import hstack

# Combine base TF-IDF features
X_train_tfidf = hstack([tfidf_item_train, tfidf_bullet_train, tfidf_desc_train])
X_test_tfidf = hstack([tfidf_item_test, tfidf_bullet_test, tfidf_desc_test])

# Add category-specific features
for cat_train, cat_test in category_tfidf_features:
    X_train_tfidf = hstack([X_train_tfidf, cat_train])
    X_test_tfidf = hstack([X_test_tfidf, cat_test])

print(f"Total TF-IDF features: {X_train_tfidf.shape[1]}")

# Store TF-IDF vectorizers for later use
tfidf_vectorizers = {
    'item': tfidf_item,
    'bullet': tfidf_bullet,
    'desc': tfidf_desc
}


Combine structured features:

In [ ]:
# Enhanced Structured Features with Categorization
print("Preparing enhanced structured features...")

# Base numerical features
base_features = [
    "log_value", "bullet_point_count", "word_count_item", "word_count_bullet", 
    "word_count_desc", "word_count_total", "char_count_total", "digit_ratio_total", 
    "avg_word_len_total"
]

# New enhanced features
enhanced_features = [
    "premium_score", "discount_score", "numeric_count", "max_numeric", "min_numeric", 
    "avg_numeric", "exclamation_count", "question_count", "uppercase_ratio", 
    "punctuation_density", "pack_size", "value_x_premium", "word_count_x_premium", 
    "pack_size_x_value"
]

# Categorical encoded features
categorical_encoded = ["unit_raw_enc", "unit_category_enc", "product_category_enc", "brand_enc"]

# Binary features
binary_features = [
    "has_multi_pack", "is_food_beverage", "is_health_beauty", "is_electronics",
    "is_weight_unit", "is_volume_unit", "is_count_unit"
]

# Keyword features
enhanced_keywords = ["pack", "organic", "premium", "bundle", "eco", "vegan", "gluten", "sugar", "diet", "mix", "instant",
                    "new", "fresh", "natural", "handmade", "limited", "sale", "discount", "special", "gift", "free",
                    "imported", "authentic", "original", "pure", "light", "soft", "strong", "durable", "high-quality",
                    "bestseller", "top-rated", "popular", "classic", "eco-friendly", "recyclable", "non-toxic", "safe",
                    "baby", "kids", "adult", "men", "women", "unisex", "winter", "summer", "travel", "portable",
                    "premium-grade", "luxury", "compact", "multi-purpose", "versatile", "handcrafted", "exclusive"]

keyword_features = [f"has_{kw}" for kw in enhanced_keywords]

# Combine all structured features
structured_features = base_features + enhanced_features + categorical_encoded + binary_features + keyword_features

print(f"Total structured features: {len(structured_features)}")

# Ensure all features exist in both train and test
missing_train = [f for f in structured_features if f not in train.columns]
missing_test = [f for f in structured_features if f not in test.columns]

if missing_train:
    print(f"Missing in train: {missing_train}")
if missing_test:
    print(f"Missing in test: {missing_test}")

# Filter to existing features
existing_features = [f for f in structured_features if f in train.columns and f in test.columns]
print(f"Using {len(existing_features)} structured features")

# Apply scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

train_struct = scaler.fit_transform(train[existing_features].fillna(0))
test_struct = scaler.transform(test[existing_features].fillna(0))

# Combine with TF-IDF features
print("Combining TF-IDF and structured features...")
X = hstack([X_train_tfidf, train_struct])
X = X.tocsr()

X_test = hstack([X_test_tfidf, test_struct])
X_test = X_test.tocsr()

y = np.log1p(train["price"].values)

print(f"Final feature matrix shape - Train: {X.shape}, Test: {X_test.shape}")
print(f"Target variable shape: {y.shape}")

# Feature importance tracking
feature_names = (
    [f"tfidf_item_{i}" for i in range(tfidf_item_train.shape[1])] +
    [f"tfidf_bullet_{i}" for i in range(tfidf_bullet_train.shape[1])] +
    [f"tfidf_desc_{i}" for i in range(tfidf_desc_train.shape[1])] +
    existing_features
)

print(f"Total features: {len(feature_names)}")

# Category-Specific Modeling Strategy

We can improve predictions by training separate models or using category-specific features for different product types, as pricing patterns may vary significantly between categories like electronics vs food items.

In [ ]:
# Category Analysis and Feature Selection
def analyze_categories_and_features():
    """Analyze how different categories behave and select important features"""
    
    print("CATEGORY DISTRIBUTION ANALYSIS")
    print("=" * 50)
    
    # Category distribution
    print("Product Category Distribution:")
    train_cat_dist = train['product_category'].value_counts()
    print(train_cat_dist)
    
    print(f"\nUnit Category Distribution:")
    train_unit_dist = train['unit_category'].value_counts()
    print(train_unit_dist)
    
    # Price analysis by category
    print(f"\nPrice Statistics by Product Category:")
    price_by_cat = train.groupby('product_category')['price'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
    print(price_by_cat)
    
    print(f"\nPrice Statistics by Unit Category:")
    price_by_unit = train.groupby('unit_category')['price'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2)
    print(price_by_unit)
    
    # Value vs Price analysis
    print(f"\nValue vs Price Correlation by Category:")
    for category in train['product_category'].unique():
        if category != 'other':
            cat_data = train[train['product_category'] == category]
            if len(cat_data) > 10:
                corr = cat_data['value_raw'].corr(cat_data['price'])
                print(f"{category:<15}: {corr:.4f} (n={len(cat_data)})")
    
    return price_by_cat, price_by_unit

# Run analysis
category_stats, unit_stats = analyze_categories_and_features()

# Feature correlation analysis for top categories
print(f"\nCORRELATION ANALYSIS FOR MAJOR CATEGORIES")
print("=" * 50)

major_cats = train['product_category'].value_counts().head(4).index
numeric_features = [f for f in existing_features if train[f].dtype in ['int64', 'float64']]

for category in major_cats:
    cat_data = train[train['product_category'] == category]
    if len(cat_data) > 20:
        print(f"\nTop features for {category} (n={len(cat_data)}):")
        cat_corr = cat_data[numeric_features + ['price']].corr()['price'].abs().sort_values(ascending=False)
        for feature, corr in cat_corr.head(8).items():
            if feature != 'price':
                print(f"  {feature:<25}: {corr:.4f}")

print(f"\nFeature preparation complete. Ready for model training!")

In [16]:
# train.head(10)
# # train.columns

LightGBM Training:

In [ ]:
# Define SMAPE function
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred) + 0.000001) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0  # avoid division by zero
    return 100 * np.mean(diff)

# K-Fold parameters
params = {
    "objective": "regression",
    "metric": "mae",  # still using MAE internally
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "random_state": 22,
}

kf = KFold(n_splits=5, shuffle=True, random_state=22)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
best_iterations = []

for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n----- Fold {fold+1} -----")

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)],
    )
    best_iterations.append(model.best_iteration)

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

    # Compute SMAPE for this fold
    fold_smape = smape(np.expm1(y_val), np.expm1(oof_preds[val_idx]))
    print(f"Fold {fold+1} SMAPE: {fold_smape:.4f}")

# Overall CV SMAPE
overall_smape = smape(np.expm1(y), np.expm1(oof_preds))
print("\nOverall CV SMAPE:", overall_smape)
average_best_iteration = int(np.mean(best_iterations))


----- Fold 1 -----
Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.515691	val's l1: 0.545055
[200]	train's l1: 0.471164	val's l1: 0.523761
[300]	train's l1: 0.442325	val's l1: 0.514657
[400]	train's l1: 0.420068	val's l1: 0.509882
[500]	train's l1: 0.401082	val's l1: 0.50742
[600]	train's l1: 0.384915	val's l1: 0.505485
[700]	train's l1: 0.370682	val's l1: 0.504158
[800]	train's l1: 0.357033	val's l1: 0.503754
[900]	train's l1: 0.345058	val's l1: 0.503523
Early stopping, best iteration is:
[893]	train's l1: 0.345795	val's l1: 0.503478
Fold 1 SMAPE: 50.8282

----- Fold 2 -----
Training until validation scores don't improve for 100 rounds
[100]	train's l1: 0.513576	val's l1: 0.551934
[200]	train's l1: 0.468672	val's l1: 0.530766
[300]	train's l1: 0.439973	val's l1: 0.522917
[400]	train's l1: 0.417371	val's l1: 0.517593
[500]	train's l1: 0.398979	val's l1: 0.514224
[600]	train's l1: 0.382408	val's l1: 0.512043


Final Model Training:

In [ ]:
# Prepare full dataset
full_train_set = lgb.Dataset(X, label=y)

# Train final model on all data
final_model = lgb.train(
    params,
    full_train_set,
    num_boost_round=average_best_iteration
)

Final model prediction:

In [ ]:
final_preds_log = final_model.predict(X_test, num_iteration=final_model.best_iteration)
final_preds = np.expm1(final_preds_log)
print(final_preds)

Code for getting data for ensembling:

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=22)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
best_iterations = []

# --- 4. Your existing K-Fold Training and Prediction Loop ---
# This loop is identical to the one you provided.
print("Starting LightGBM K-Fold Training...")
for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n----- Fold {fold+1} -----")

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y[trn_idx], y[val_idx]

    train_set = lgb.Dataset(X_tr, label=y_tr)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

    model = lgb.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)],
    )
    best_iterations.append(model.best_iteration)

    oof_preds[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / kf.n_splits

print("\nK-Fold training complete. Saving predictions...")

# Save the out-of-fold predictions for the training set
oof_df = pd.DataFrame({'sample_id': train['sample_id'], 'lgbm_oof_preds': oof_preds})
oof_df.to_csv('lgbm_oof_preds.csv', index=False)

# Save the averaged predictions for the test set
test_preds_df = pd.DataFrame({'sample_id': test['sample_id'], 'lgbm_test_preds': test_preds})
test_preds_df.to_csv('lgbm_test_preds.csv', index=False)

print("OOF and test predictions saved successfully.")
print(f"OOF predictions shape: {oof_df.shape}")
print(f"Test predictions shape: {test_preds_df.shape}")

Save predictions:

In [ ]:
submission = pd.DataFrame({
    "sample_id": test["sample_id"],
    "price": final_preds
})

submission.to_csv(r"predictions_lgbm.csv", index=False)
print(submission.head())

In [ ]:
import joblib
import pickle

print("Saving enhanced model and feature engineering components...")

# Save the final model
joblib.dump(final_model, "enhanced_lgbm_model.pkl")

# Save all feature engineering components
feature_engineering_components = {
    'tfidf_vectorizers': tfidf_vectorizers,
    'label_encoders': label_encoders,
    'scaler': scaler,
    'feature_names': feature_names,
    'existing_features': existing_features,
    'enhanced_keywords': enhanced_keywords,
    'unit_categories': UNIT_CATEGORIES,
    'product_categories': PRODUCT_CATEGORIES,
    'brand_patterns': BRAND_PATTERNS
}

with open("enhanced_feature_components.pkl", "wb") as f:
    pickle.dump(feature_engineering_components, f)

# Save category analysis results
category_analysis = {
    'category_stats': category_stats,
    'unit_stats': unit_stats,
    'major_categories': major_cats if 'major_cats' in locals() else []
}

with open("category_analysis.pkl", "wb") as f:
    pickle.dump(category_analysis, f)

print("✓ Saved enhanced_lgbm_model.pkl")
print("✓ Saved enhanced_feature_components.pkl") 
print("✓ Saved category_analysis.pkl")

print(f"\nModel Performance Summary:")
print(f"- Total features: {X.shape[1]}")
print(f"- TF-IDF features: {X_train_tfidf.shape[1]}")  
print(f"- Structured features: {len(existing_features)}")
print(f"- Product categories: {len(train['product_category'].unique())}")
print(f"- Unit categories: {len(train['unit_category'].unique())}")

# Create a simple inference function for later use
def create_inference_function():
    """Create a function for making predictions on new data"""
    
    inference_code = '''
def predict_price_enhanced(catalog_content, model_path="enhanced_lgbm_model.pkl", 
                          components_path="enhanced_feature_components.pkl"):
    """
    Predict price for new catalog content using enhanced features
    """
    import joblib
    import pickle
    import pandas as pd
    import numpy as np
    from scipy.sparse import hstack
    
    # Load model and components
    model = joblib.load(model_path)
    with open(components_path, "rb") as f:
        components = pickle.load(f)
    
    # Create dataframe
    df = pd.DataFrame({"catalog_content": [catalog_content]})
    
    # Apply enhanced feature engineering
    df = enhanced_feature_engineering(df, is_train=False, correlation_analysis=False)
    
    # Apply encodings and scaling
    for feature in components['label_encoders'].keys():
        if feature in df.columns:
            le = components['label_encoders'][feature]
            df[f"{feature}_enc"] = le.transform(df[feature].astype(str).fillna('unknown'))
    
    # TF-IDF transformation
    tfidf_item = components['tfidf_vectorizers']['item'].transform(df["item_name"])
    tfidf_bullet = components['tfidf_vectorizers']['bullet'].transform(df["bullet_points"])
    tfidf_desc = components['tfidf_vectorizers']['desc'].transform(df["product_desc"])
    X_tfidf = hstack([tfidf_item, tfidf_bullet, tfidf_desc])
    
    # Structured features
    X_struct = components['scaler'].transform(df[components['existing_features']].fillna(0))
    
    # Combine features
    X = hstack([X_tfidf, X_struct]).tocsr()
    
    # Predict
    pred_log = model.predict(X)[0]
    price = np.expm1(pred_log)
    
    return price
    '''
    
    with open("enhanced_inference.py", "w") as f:
        f.write(inference_code)
    
    print("✓ Created enhanced_inference.py")

create_inference_function()

print("\nEnhanced model training and saving complete! 🚀")